In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# MRI Reconstruction — Results Visualization\n", "Compare architectures, visualize reconstructions, and plot training curves."]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys, os\n",
    "sys.path.insert(0, '..')\n",
    "\n",
    "import torch\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import json\n",
    "\n",
    "from models.unet     import UNet, count_parameters\n",
    "from models.bt_unet  import BTUNet\n",
    "from models.swinunet import SwinUNet\n",
    "\n",
    "plt.style.use('seaborn-v0_8-dark')\n",
    "%matplotlib inline\n",
    "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n",
    "print(f'Device: {device}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 1. Architecture Overview"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "configs = [\n",
    "    ('U-Net Baseline', UNet(in_channels=1, out_channels=1, base_ch=32, n_levels=4)),\n",
    "    ('BT-UNet',        BTUNet(in_channels=1, out_channels=1, base_ch=32, n_levels=4)),\n",
    "    ('SwinUNet',       SwinUNet(img_size=320, patch_size=4, in_ch=1, out_ch=1, embed_dim=64, ws=8, n_levels=3)),\n",
    "]\n",
    "\n",
    "dummy = torch.randn(1, 1, 320, 320)\n",
    "print(f'  {\"Architecture\":<20} {\"Params (M)\":>10} {\"Output Shape\":>16}')\n",
    "print('  ' + '─' * 50)\n",
    "for name, model in configs:\n",
    "    model.eval()\n",
    "    with torch.no_grad():\n",
    "        out = model(dummy)\n",
    "    n = count_parameters(model)\n",
    "    print(f'  {name:<20} {n/1e6:>10.1f} {str(tuple(out.shape)):>16}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 2. Training Curves (from saved history)"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Loads JSON history files saved during training\n",
    "# Run training first: python main.py train --model swinunet\n",
    "\n",
    "model_names = ['unet', 'bt_unet', 'swinunet']\n",
    "histories   = {}\n",
    "for name in model_names:\n",
    "    path = f'../checkpoints/{name}_history.json'\n",
    "    if os.path.exists(path):\n",
    "        with open(path) as f:\n",
    "            histories[name] = json.load(f)\n",
    "\n",
    "if histories:\n",
    "    fig, axes = plt.subplots(1, 3, figsize=(15, 4))\n",
    "    colors = {'unet': 'steelblue', 'bt_unet': 'coral', 'swinunet': 'mediumseagreen'}\n",
    "\n",
    "    for name, hist in histories.items():\n",
    "        ep = range(1, len(hist['val_loss']) + 1)\n",
    "        c  = colors.get(name, 'gray')\n",
    "        axes[0].plot(ep, hist['val_loss'],  color=c, label=name)\n",
    "        axes[1].plot(ep, hist['val_psnr'],  color=c, label=name)\n",
    "        axes[2].plot(ep, hist['val_ssim'],  color=c, label=name)\n",
    "\n",
    "    for ax, title, ylabel in zip(axes,\n",
    "        ['Validation Loss', 'Validation PSNR (dB)', 'Validation SSIM'],\n",
    "        ['Loss', 'PSNR (dB)', 'SSIM']):\n",
    "        ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)\n",
    "        ax.legend()\n",
    "\n",
    "    plt.suptitle('Training Curves — Architecture Comparison', fontsize=13)\n",
    "    plt.tight_layout(); plt.show()\n",
    "else:\n",
    "    print('No training history found. Train models first with main.py train ...')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 3. Reconstruction Visualization"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load dataset and best checkpoint for visualization\n",
    "# Requires: data/knee_singlecoil_val/ and checkpoints/swinunet_best.pt\n",
    "\n",
    "VAL_DIR  = '../data/knee_singlecoil_val'\n",
    "CKPT     = '../checkpoints/swinunet_best.pt'\n",
    "\n",
    "if os.path.exists(VAL_DIR) and os.path.exists(CKPT):\n",
    "    from data.preprocessing import FastMRIKneeDataset\n",
    "    from training.evaluate  import load_model, visualize_reconstructions\n",
    "\n",
    "    model   = load_model('swinunet', CKPT, device)\n",
    "    dataset = FastMRIKneeDataset(VAL_DIR)\n",
    "    visualize_reconstructions(model, dataset, device, n_examples=4,\n",
    "                               save_path='reconstructions_nb.png')\n",
    "    from IPython.display import Image\n",
    "    display(Image('reconstructions_nb.png'))\n",
    "else:\n",
    "    print('Download the fastMRI dataset and train a model first.')\n",
    "    print(f'  Missing: {VAL_DIR if not os.path.exists(VAL_DIR) else CKPT}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 4. Performance Summary Table"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Results from paper\n",
    "results = {\n",
    "    'U-Net Baseline': {'Val Loss': 0.0496, 'PSNR (dB)': 28.03, 'SSIM': 0.6935},\n",
    "    'BT-UNet':        {'Val Loss': 0.0412, 'PSNR (dB)': 29.87, 'SSIM': 0.7102},\n",
    "    'SwinUNet':       {'Val Loss': 0.0352, 'PSNR (dB)': 33.10, 'SSIM': 0.7274},\n",
    "}\n",
    "pd_results = pd.DataFrame(results).T\n",
    "pd_results.style.highlight_max(axis=0, subset=['PSNR (dB)', 'SSIM'], color='lightgreen')\\\n",
    "                .highlight_min(axis=0, subset=['Val Loss'], color='lightgreen')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python", "version": "3.11.0"}
 },
 "nbformat": 4,
 "nbformat_minor": 4
}